In [1]:
import glob
import os

print("--- Searching for Captions ---")
# The '**' tells glob to search recursively through all subdirectories
possible_paths = glob.glob('/kaggle/input/**/captions', recursive=True)

if possible_paths:
    print(f"SUCCESS! Found captions at: {possible_paths[0]}")
else:
    print("FAILED: No 'captions' folder found.\n")
    print("--- Mapping /kaggle/input/ Directory Structure ---")
    for root, dirs, files in os.walk('/kaggle/input/'):
        # Limit depth to 2 levels so it doesn't print thousands of image files
        depth = root.count(os.sep) - '/kaggle/input/'.count(os.sep)
        if depth < 2:
            print(f"DIR: {root}")
            if dirs:
                print(f"  Subfolders: {dirs}")

--- Searching for Captions ---
SUCCESS! Found captions at: /kaggle/input/notebooks/markosumire/onlycaptioningcommit/captions


In [2]:
# Install dependencies
!pip install -q timm pytorch-msssim lpips scikit-image thop gdown transformers accelerate

import os
import shutil
import subprocess
import json

# --- 1. EXACT MOUNTED CAPTIONS PATH ---
MOUNTED_CAPTIONS_BASE = '/kaggle/input/notebooks/markosumire/onlycaptioningcommit/captions'

# 2. Clone repository
!git clone https://github.com/sumire25/RSHazeNet.git /kaggle/working/RSHazeNet
%cd /kaggle/working/RSHazeNet

# 3. Setup RRSHID
RRSHID_ZIP = '/kaggle/working/RRSHID.zip'
RRSHID_BASE = '/kaggle/working/RRSHID'
if not os.path.isdir(RRSHID_BASE):
    !wget -q -O {RRSHID_ZIP} "https://github.com/lwCVer/RRSHID/releases/download/dataset/RRSHID.zip"
    !unzip -q {RRSHID_ZIP} -d {RRSHID_BASE}
    !rm -f {RRSHID_ZIP}

# 4. Virtual Dataset Aggregator (VLM Version with Caption Merging)
UNIFIED_SATEHAZE = '/kaggle/working/Unified_SateHaze1k'
UNIFIED_RRSHID = '/kaggle/working/Unified_RRSHID'

def prepare_unified_layout(source_base, unified_base, levels, splits, original_base_for_key):
    if os.path.exists(unified_base): shutil.rmtree(unified_base)
    
    for split in splits:
        hazy_dir = os.path.join(unified_base, split, 'hazy')
        gt_dir = os.path.join(unified_base, split, 'GT')
        os.makedirs(hazy_dir, exist_ok=True)
        os.makedirs(gt_dir, exist_ok=True)
        
        master_captions = {}
        
        for level in levels:
            raw_split = os.path.join(source_base, level, split)
            if not os.path.isdir(raw_split): continue
                
            raw_hazy = os.path.join(raw_split, 'hazy')
            raw_gt = next((os.path.join(raw_split, c) for c in ['GT', 'gt', 'clear'] if os.path.isdir(os.path.join(raw_split, c))), None)
            
            if not raw_gt or not os.path.isdir(raw_hazy): continue
            
            # Symlink images
            for fname in os.listdir(raw_hazy):
                if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')): continue
                unique_fname = f"{level}_{fname}"
                os.symlink(os.path.join(raw_hazy, fname), os.path.join(hazy_dir, unique_fname))
                os.symlink(os.path.join(raw_gt, fname), os.path.join(gt_dir, unique_fname))
                
            # Merge JSON metadata
            cap_key = f"{original_base_for_key}/{level}/{split}/hazy".replace('/', '_').strip('_')
            cap_path = os.path.join(MOUNTED_CAPTIONS_BASE, cap_key, 'captions.json')
            
            if os.path.exists(cap_path):
                with open(cap_path, 'r') as f:
                    level_caps = json.load(f)
                    for k, v in level_caps.items():
                        master_captions[f"{level}_{k}"] = v
        
        # Save fused metadata directly to the dataloader's target directory
        if master_captions:
            with open(os.path.join(hazy_dir, 'captions.json'), 'w') as f:
                json.dump(master_captions, f, indent=4)

prepare_unified_layout('/kaggle/input/datasets/xuxingxing233/satehaze1k', UNIFIED_SATEHAZE, 
                       ['Haze1k_thin', 'Haze1k_moderate', 'Haze1k_thick'], ['train', 'test'], 
                       '/kaggle/input/datasets/xuxingxing233/satehaze1k')

prepare_unified_layout(RRSHID_BASE, UNIFIED_RRSHID, 
                       ['thin_fog', 'moderate_fog', 'thick_fog'], ['train', 'val', 'test'], 
                       '/kaggle/working/RRSHID')

# 5. Train Orchestrator
def run_vlm_train(dataset_name, unified_base, val_split):
    print(f"\n{'='*50}\nTraining VLM-Guided Pipeline: {dataset_name}\n{'='*50}")
    
    save_dir = f'/kaggle/working/weights/{dataset_name}_VLM'
    os.makedirs(save_dir, exist_ok=True)
    
    train_cmd = [
        'python', 'train.py',
        '--train_input', f'{unified_base}/train/hazy',
        '--train_target', f'{unified_base}/train/GT',
        '--val_input', f'{unified_base}/{val_split}/hazy',
        '--val_target', f'{unified_base}/{val_split}/GT',
        '--caption',
        '--epochs', '100',
        '--batch_size_train', '8',
        '--patch_size_train', '256' if dataset_name == "RRSHID" else '512',
        '--patch_size_val', '256' if dataset_name == "RRSHID" else '512',
        '--save_dir', save_dir
    ]
    
    checkpoint_path = f'{save_dir}/model_best.pth'
    if os.path.exists(checkpoint_path):
        print(f"--> Found existing checkpoint at {checkpoint_path}. Skipping/Resuming training!")
        train_cmd += ['--resume', checkpoint_path]
        
    print("-> Training...")
    subprocess.run(train_cmd, check=True)

# Execute isolated training logic
run_vlm_train("SateHaze1k", UNIFIED_SATEHAZE, "test")
run_vlm_train("RRSHID", UNIFIED_RRSHID, "val")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 861.5 kB/s eta 0:00:00
Cloning into '/kaggle/working/RSHazeNet'...
remote: Enumerating objects: 107, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 107 (delta 57), reused 75 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (107/107), 2.86 MiB | 10.01 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/kaggle/working/RSHazeNet

Training VLM-Guided Pipeline: SateHaze1k
-> Training...


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 2366.69it/s, Materializing param=visual_projection.weight]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/kaggle/working/RSHazeNet/train.py:178: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


[Caption] Loaded 958 train captions, 135 val captions.
-------------------------------------------------------------------------------------------------------
[Caption] Conditioning enabled (CLIP=openai/clip-vit-base-patch32, haze_weight=0.0).
  0%|          | 0/119 [00:00<?, ?it/s]

/kaggle/working/RSHazeNet/train.py:199: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(True):


Training !!!  Epoch 1 / 100,  Batch Loss 0.040489: 100%|██████████| 119/119 [02:48<00:00,  1.42s/it]

[Val Epoch 1] PSNR: 22.5713 | SSIM: 0.8097 | SAM: 0.7798 | ERGAS: 10.2705
------------------------------------------------------------
Epoch:  1  Finished,  Time:  181.5411 s,  Loss:  0.030362, best psnr:  22.571.
-------------------------------------------------------------------------------------------------------
Training !!!  Epoch 2 / 100,  Batch Loss 0.031338: 100%|██████████| 119/119 [02:11<00:00,  1.10s/it]
------------------------------------------------------------
Epoch:  2  Finished,  Time:  131.2716 s,  Loss:  0.025604, best psnr:  22.571.
-------------------------------------------------------------------------------------------------------
Training !!!  Epoch 3 / 100,  Batch Loss 0.021471: 100%|██████████| 119/119 [02:11<00:00,  1.10s/it]

[Val Epoch 3] PSNR: 24.6888 | SSIM: 0.8855 | SAM: 0.7932 | ERGAS: 8.0093
------------------------------------------------------------

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 2335.89it/s, Materializing param=visual_projection.weight]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/kaggle/working/RSHazeNet/train.py:178: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


[Caption] Loaded 2441 train captions, 308 val captions.
-------------------------------------------------------------------------------------------------------
[Caption] Conditioning enabled (CLIP=openai/clip-vit-base-patch32, haze_weight=0.0).
  0%|          | 0/305 [00:00<?, ?it/s]

/kaggle/working/RSHazeNet/train.py:199: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(True):


Training !!!  Epoch 1 / 100,  Batch Loss 0.009458: 100%|██████████| 305/305 [02:00<00:00,  2.54it/s]

[Val Epoch 1] PSNR: 24.7173 | SSIM: 0.7729 | SAM: 1.1264 | ERGAS: 9.7508
------------------------------------------------------------
Epoch:  1  Finished,  Time:  129.2751 s,  Loss:  0.020489, best psnr:  24.717.
-------------------------------------------------------------------------------------------------------
Training !!!  Epoch 2 / 100,  Batch Loss 0.012948: 100%|██████████| 305/305 [01:27<00:00,  3.47it/s]
------------------------------------------------------------
Epoch:  2  Finished,  Time:  87.9337 s,  Loss:  0.017312, best psnr:  24.717.
-------------------------------------------------------------------------------------------------------
Training !!!  Epoch 3 / 100,  Batch Loss 0.011210: 100%|██████████| 305/305 [01:27<00:00,  3.48it/s]

[Val Epoch 3] PSNR: 25.2413 | SSIM: 0.7676 | SAM: 1.2886 | ERGAS: 9.2575
------------------------------------------------------------
E